In [1]:
import re
import pandas as pd
from collections import defaultdict

## Multiome dataset from Erickson et al 2026

In [ ]:
df = pd.read_csv('../data/0_SupplementaryTable_Links_11062025.csv', header=0, index_col=0, sep=',')

In [ ]:
df

,key,gene,Trait,Variation,Position,name,ChIP_CS17,H3K27ac_AV,CAGE,DAR_coarse,Vista_region,DAR_granular,ench_5000_JW,candidate_ench_JW,DAR_regional,PhastCons_Avg,HiC_category,DAR_unique_coarse_regional
1,chr1-100354714-100355634,COL11A1,NaN,NaN,NaN,"14_EnhA2,18_EnhAc,10_TxEnh5'","chr1-100354444-100355244,chr1-100355244-100355...",chr1-100349798-100357419,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.181115,No Loop,NaN
2,chr1-100362608-100363547,COL11A1,NaN,NaN,NaN,25_Quies,chr1-100356844-100386844,NaN,chr1-100362870-100363265,NaN,NaN,NaN,NaN,NaN,NaN,0.081193,No Loop,NaN
3,chr1-100386980-100388445,COL11A1,NaN,NaN,NaN,"18_EnhAc,25_Quies","chr1-100386844-100387644,chr1-100387644-100414644",chr1-100386425-100389125,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.049219,No Loop,NaN
4,chr1-100427960-100428858,COL11A1,NaN,NaN,NaN,25_Quies,chr1-100414844-100459044,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.068037,No Loop,NaN
5,chr1-1004957-1005973,HES4,NaN,NaN,NaN,"14_EnhA2,17_EnhW2,15_EnhAF","chr1-1004820-1005420,chr1-1005420-1005620,chr1...",NaN,NaN,"Myoblast,Perivascular",NaN,"myoblast,pericyte",NaN,NaN,NaN,0.005465,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50285,chrX-9728936-9729917,GPR143,NaN,NaN,NaN,25_Quies,chrX-9727760-9764560,NaN,NaN,NaN,NaN,melanocytes,NaN,NaN,NaN,0.024134,Border,NaN
50286,chrX-9745204-9746092,SHROOM2,NaN,NaN,NaN,25_Quies,chrX-9727760-9764560,NaN,NaN,Endothelia,NaN,endothelia,NaN,NaN,NaN,0.068668,Border,NaN
50287,chrX-9746101-9747052,SHROOM2,NaN,NaN,NaN,25_Quies,chrX-9727760-9764560,NaN,NaN,Endothelia,NaN,"endothelia,melanocytes",NaN,NaN,NaN,0.066104,Border,NaN
50288,chrX-97461857-97462902,PCDH19,NaN,NaN,NaN,25_Quies,chrX-96956201-97563401,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.752081,No Loop,NaN


In [ ]:
#extracting only annotated enhancers with reg exp
rx = re.compile(r'Enh')
mask = df['name'].astype(str).apply(lambda s: bool(rx.search(s)))
filtered = df.loc[mask, :]

In [7]:
#annotation each enhancer coordinate by assigned genes
counts = defaultdict(int)
pairs = {}

for _, r in filtered[['gene', 'key']].iterrows():
    g = str(r['gene'])
    
    if ',' in g:
       g = g.replace(',', '_') #sometime more then 1 gene correlates with enhancer, they are separated by ','
    
    counts[g] += 1
    enh_name = f"{g}_en{counts[g]}"

    chrom, start, end = r['key'].split('-')
    pairs[(chrom, int(start), int(end))] = enh_name

annotated_enh = (
    pd.Series(pairs, name="genes")
    .rename_axis(['chr', 'start', 'stop'])
    .reset_index()
)

In [8]:
annotated_enh

,chr,start,stop,genes
0,chr1,100354714,100355634,COL11A1_en1
1,chr1,100386980,100388445,COL11A1_en2
2,chr1,1004957,1005973,HES4_en1
3,chr1,1024412,1025669,PRDM16_en1
4,chr1,103283899,103284941,COL11A1_VCAM1_en1
...,...,...,...,...
11772,chrX,69153085,69154020,KIF4A_en1
11773,chrX,71181717,71182659,KIF4A_en2
11774,chrX,72004050,72004998,NHSL2_en1
11775,chrX,72394151,72395111,EDA_en1


In [10]:
#saving coordinates and it's annotation
annotated_enh.to_csv('../data/0_enhancer_gene_pairs.csv', sep=';', header=True, index=False)

## GWAS dataset from Goovaerts et al 2026

In [11]:
hg19_df = pd.read_csv('../data/0_Goovaert_2026.csv', sep=';', header=0)
hg19_df

,chr,pos_hg19,rs,p,neg_log_p,maf,n,a1,a2,genomic_locus,best_segment,primary_anatomical_region,other_anatomical_regions,other_snps_within_1mb,genes_great_distance_to_tss
0,1,978603,rs780665359,6.625200e-11,"10,1788","0,4387",48337,C,CCT,1,whole_craniofacial,Whole craniofacial region,Cranial vault,1,"AGRN (+23100), RNF223 (+31084)"
1,1,1687152,rs4648815,3.663500e-10,"9,4361","0,4754",48770,A,G,2,whole_craniofacial,Whole craniofacial region,Cranial vault,2,"SLC35E2 (-9721), NADK (+22757)"
2,1,2201744,rs77409011,2.423800e-12,"11,6155","0,0704",50593,T,G,3,whole_craniofacial,Whole craniofacial region,"Face, Lower face and midface, Nose, Upper face",5,"RER1 (-121528), SKI (+41610)"
3,1,2761375,rs12049543,3.036000e-19,"18,5177","0,4235",49902,A,C,4,whole_craniofacial,Whole craniofacial region,"Orbital region, Cranial vault, Face, Upper fac...",6,"MMEL1 (-196946), ACTRT2 (-176671)"
4,1,2798712,rs4373707,8.804400e-108,"107,0553","0,4660",49716,C,T,4,whole_craniofacial,Whole craniofacial region,"Cranial vault, Anterior cranial vault, Face, P...",6,"MMEL1 (-234283), ACTRT2 (-139334)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2574,23,135972905,rs5931014,1.403500e-08,"7,8528","0,4580",50233,A,G,1172,whole_craniofacial,Whole craniofacial region,NaN,2,"RBMX (-10021), GPR101 (+140928)"
2575,23,136955404,rs4829906,5.819700e-241,"240,2351","0,4196",50132,A,G,1173,whole_craniofacial,Whole craniofacial region,"Cranial vault, Anterior cranial vault, Posteri...",2,ZIC3 (+307103)
2576,23,136958406,rs6633933,6.768600e-167,"166,1695","0,4769",50144,T,C,1173,whole_craniofacial,Whole craniofacial region,"Cranial vault, Posterior cranial vault, Anteri...",2,ZIC3 (+310105)
2577,23,153582703,rs2070827,6.071600e-11,"10,2167","0,1119",50555,A,G,1174,whole_craniofacial,Whole craniofacial region,"Upper face, Orbital region",1,"FLNA (+20291), TKTL1 (+58679)"


In [5]:
# hg19 --> hg38 chain file

!wget -O '../data/hg19Tohg38.over.chain.gz' ftp://hgdownload.soe.ucsc.edu/goldenPath/hg19/liftOver/hg19ToHg38.over.chain.gz --timeout=1 --tries=5

--2026-03-23 10:10:03--  ftp://hgdownload.soe.ucsc.edu/goldenPath/hg19/liftOver/hg19ToHg38.over.chain.gz
           => ‘../data/hg19Tohg38.over.chain.gz’
Resolving hgdownload.soe.ucsc.edu (hgdownload.soe.ucsc.edu)... 128.114.119.163
Connecting to hgdownload.soe.ucsc.edu (hgdownload.soe.ucsc.edu)|128.114.119.163|:21... connected.
Logging in as anonymous ... Logged in!
==> SYST ... done.    ==> PWD ... done.
==> TYPE I ... done.  ==> CWD (1) /goldenPath/hg19/liftOver ... done.
==> SIZE hg19ToHg38.over.chain.gz ... 227698
==> PASV ... done.    ==> RETR hg19ToHg38.over.chain.gz ... done.
Length: 227698 (222K) (unauthoritative)

hg19ToHg38.over.cha 100%[===================>] 222.36K   336KB/s    in 0.7s    

2026-03-23 10:10:06 (336 KB/s) - ‘../data/hg19Tohg38.over.chain.gz’ saved [227698]



In [ ]:
# saving hg19 coordinates as a BED file (as a lift over input)
hg19_pos = hg19_df.iloc[:, 0:2].copy()
hg19_pos.columns = ['chr', 'pos_hg19']

hg19_pos['end'] = hg19_pos['pos_hg19'] + 1
hg19_pos['chr'] = 'chr' + hg19_pos['chr'].astype(str)

hg19_pos['snp_coord'] = (
    hg19_pos['chr'].astype(str) + ":" +
    hg19_pos['pos_hg19'].astype(str)
)

hg19_pos.to_csv('../data/0_hg19_SNPs.coord.bed', sep='\t', header=None, index=False)


In [29]:
# hg19 --> hg38 lift over

!/home/nadja/miniforge3/envs/internship_workflow/bin/liftOver -minMatch=0.9 '../data/0_hg19_SNPs.coord.bed' '../data/hg19Tohg38.over.chain.gz' '../data/0_hg38_SNPs.bed' '../data/0_unmapped_hg38_SNPs.bed' 

Reading liftover chains
Mapping coordinates


## STRING background

To generate a background for gene enrichment analysis, we exract all gene names from the supplementary table. The table contains all genes which have any expression correlation with open chromatin regions.

In [ ]:
df = pd.read_csv('../data/0_SupplementaryTable_Links_11062025.csv', sep=',', header=0)

df_genes = df.loc[:, "gene"]

string_genes = set()
for genes in df_genes.dropna():
    for g in str(genes).split(","):
        g = g.strip()
        if g:
            string_genes.add(g)

In [ ]:
with open('../data/0_STRING_background_genes.txt', 'w') as f:
    for gene in string_genes:
        f.write(f'{gene}\n')